<a href="https://colab.research.google.com/github/sahasraa178/Machine-Learning-sem-4/blob/main/mllabassisment9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MACHINE LEARNING LAB ASSIGNMENT 9**


BL.SC.U4AIE24009


B.SAHASRAA

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving BERT_Embeddings.xlsx to BERT_Embeddings.xlsx


A1

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- Load dataset ---
df = pd.read_excel("BERT_Embeddings.xlsx")
X = df.drop(columns=['label','Student','Teacher']).values
y = df['label'].values

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Base models ---
base_models = [
    ('lr', LogisticRegression(max_iter=500)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingClassifier(random_state=42))
]

# --- Meta model ---
final_estimator = LogisticRegression(max_iter=500)

stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=final_estimator,
    passthrough=True
)

stack_clf.fit(X_train, y_train)
y_pred_stack = stack_clf.predict(X_test)

print("=== A1: Stacking Classifier ===")
print("Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_stack))
print("Classification Report:\n", classification_report(y_test, y_pred_stack))


=== A1: Stacking Classifier ===
Accuracy: 0.5354449472096531
Confusion Matrix:
 [[ 15  31  46]
 [  6  81 126]
 [ 11  88 259]]
Classification Report:
               precision    recall  f1-score   support

           1       0.47      0.16      0.24        92
           2       0.41      0.38      0.39       213
           3       0.60      0.72      0.66       358

    accuracy                           0.54       663
   macro avg       0.49      0.42      0.43       663
weighted avg       0.52      0.54      0.51       663



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from xgboost import XGBClassifier


df = pd.read_excel("BERT_Embeddings.xlsx")


X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


pca = PCA(n_components=200)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=9, weights='distance')),

    ('svm', SVC(
        C=2.0,
        kernel='rbf',
        gamma='scale',
        probability=True
    )),

    ('rf', RandomForestClassifier(
        n_estimators=150,
        max_depth=None,
        random_state=42
    )),

    ('xgb', XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss'
    )),

    ('mlp', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=400,
        learning_rate_init=0.001
    ))
]


meta_model = LogisticRegression(max_iter=1000)

model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=-1
)



model.fit(X_train, y_train)



y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("High Accuracy Stacking:", accuracy)

High Accuracy Stacking: 0.9743589743589743


A2

without stacking

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_excel("BERT_Embeddings.xlsx")

# Prepare features
X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=200)),
    ('classifier', RandomForestClassifier(n_estimators=100))
])

# Train
pipeline.fit(X_train, y_train)

# Test
y_pred = pipeline.predict(X_test)

# Result
print("Pipeline Accuracy:", accuracy_score(y_test, y_pred))

Pipeline Accuracy: 0.6259426847662142


with stacking

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Load dataset
df = pd.read_excel("BERT_Embeddings.xlsx")

# Prepare features
X = df.select_dtypes(include=['number'])
y = df["label"]
X.columns = X.columns.astype(str)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Base models
base_models = [
    ('knn', KNeighborsClassifier(n_neighbors=9, weights='distance')),
    ('svm', SVC(C=2.0, probability=True)),
    ('rf', RandomForestClassifier(n_estimators=100)),
    ('xgb', XGBClassifier(n_estimators=100, eval_metric='logloss')),
    ('mlp', MLPClassifier(max_iter=300))
]

# Meta model
meta_model = LogisticRegression(max_iter=1000)

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    n_jobs=-1
)

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=200)),
    ('stacking', stacking_model)
])

# Train
pipeline.fit(X_train, y_train)

# Test
y_pred = pipeline.predict(X_test)

# Result
print("Pipeline + Stacking Accuracy:", accuracy_score(y_test, y_pred))

Pipeline + Stacking Accuracy: 0.9668174962292609


A3

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=167f41f344e538c2e554eae70c100b4ed191e82a373d3606d3a179466599ff93
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# converting data
X_train_np = np.array(X_train)
X_test_np = np.array(X_test)

# creating explainer
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=[str(i) for i in range(X_train_np.shape[1])],
    class_names=['1', '2', '3'],
    mode='classification'
)

# selecting sample
sample = X_test_np[0]

# explanation
exp = explainer.explain_instance(
    data_row=sample,
    predict_fn=pipeline.predict_proba
)


print(exp.as_list())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


[('2.00 < 768 <= 3.00', -0.4023377489973863), ('756 > 0.30', -0.03834022752166851), ('178 <= -0.04', -0.03224599572844384), ('-0.13 < 151 <= -0.02', 0.02954547603309418), ('316 <= -0.09', 0.028756663337117072), ('632 <= -0.14', -0.027960690344016978), ('79 <= -0.01', 0.02466063868825068), ('220 <= 0.09', 0.023130025849990687), ('-0.10 < 23 <= -0.00', -0.022595586436250308), ('397 > 0.08', 0.018396935502236537)]
